# Example script for Hackathon

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have a limited chances of making queries in total.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

## 1. collect training data

Upload `sequence.fasta`, `train.csv`, and `test.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [3]:
with open('Data/sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [4]:
len(sequence_wt)

656

In [5]:
def get_mutated_sequence(mut, sequence_wt):
  wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

  sequence = deepcopy(sequence_wt)

  return sequence[:pos]+mt+sequence[pos+1:]

In [7]:
df_train = pd.read_csv('Data/train.csv')
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_train

,mutant,DMS_score,sequence
0,M0Y,0.2730,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.2857,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.2153,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.3122,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.2180,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...,...
1135,P347D,0.3876,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1136,P347C,0.1837,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1137,P347A,0.4611,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1138,P347M,0.2412,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [9]:
df_test = pd.read_csv('Data/test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [10]:
# TODO: integrate the query data that you acquired each round into df_train

## 2. Train a prediction model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

In [11]:
'''hyperparameters'''

seq_length = 656
seed = 0 # seed for splitting the validation set
val_ratio = 0.2 # proportion of validation set

In [12]:
class ProteinDataset(Dataset):
    def __init__(self, df, istrain=True):

        alphabet = 'ACDEFGHIKLMNPQRSTVWY'
        map_a2i = {j:i for i,j in enumerate(alphabet)}
        map_i2a = {i:j for i,j in enumerate(alphabet)}

        self.df = df

        self.num_samples = len(self.df)
        self.seq_length = len(self.df.sequence.values[0])
        self.num_channels = 20

        # TODO: replace one-hot encodings with your own encodings
        self.encodings = np.zeros((self.num_samples, self.num_channels, self.seq_length)).astype(np.float32)
        self.targets = np.zeros(self.num_samples).astype(np.float32)

        if istrain:
          for it, (seq,target) in enumerate(self.df[['sequence', 'DMS_score']].values):
              for i,aa in enumerate(seq):
                  self.encodings[it,map_a2i[aa],i] = 1
              self.targets[it] = target

          self.encodings = self.encodings.astype(np.float32)
          self.targets = self.targets.astype(np.float32)
        else:
          for it, seq in enumerate(self.df['sequence'].values):
              for i,aa in enumerate(seq):
                  self.encodings[it,map_a2i[aa],i] = 1

          self.encodings = self.encodings.astype(np.float32)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return torch.tensor(self.encodings[idx]), torch.tensor(self.targets[idx])

In [13]:
train_dataset = ProteinDataset(df_train)
test_dataset = ProteinDataset(df_test, istrain=False)

# split validation set
train_dataset, val_dataset = train_test_split(train_dataset, test_size=val_ratio, random_state=seed, shuffle=True)

# TODO: revise according to your own model
train_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

In [14]:
# TODO: build your own prediction model to replace linear regression
# Hint: don't forget to use the validation set: you can either integrate the validation data into the training set or use it separately for early stopping

X_train, y_train = next(iter(train_loader))

regressor = LinearRegression()
regressor.fit(X_train.view(X_train.size(0), -1).numpy(), y_train.numpy())

X_test, _ = next(iter(test_loader))
y_test_pred = regressor.predict(X_test.view(X_test.size(0), -1).numpy())

In [15]:
df_test['DMS_score_predicted'] = y_test_pred
df_test

,mutant,sequence,DMS_score_predicted
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223193
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
...,...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188


In [16]:
df_test[['mutant', 'DMS_score_predicted']].to_csv('test_predictions.csv')

## 3. Select query for next round

In [17]:
df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

,mutant,sequence,DMS_score_predicted
1965,S108A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223193
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223193
493,K26A,MVNEARGNSSLNPCLEGSASSGSESSADSSRCSTPGLDPERHERLR...,0.223191
111,G6A,MVNEARANSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223191
1008,G55A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223190
...,...,...,...
5181,A327H,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223189
5182,A327G,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223189
8571,Y511A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223189
5184,A327E,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223189


In [18]:
# Example: randomly select 100 test variants to be queried.
# Note: random selection may not be a good strategy
# TODO: select query mutants for the next round based on your own criteria

querys = np.random.choice(df_test.mutant.values, size=100, replace=False)
querys


array(['G186W', 'S372F', 'R67E', 'N172I', 'I111S', 'P384T', 'A654R',
       'N443S', 'I568T', 'H41T', 'D91R', 'S54F', 'L125C', 'E105W',
       'Y511L', 'S113V', 'F64H', 'I487V', 'A84Y', 'S493K', 'L553T',
       'V217K', 'N629P', 'E519K', 'A654E', 'N7W', 'R369G', 'F63I',
       'V178L', 'K400V', 'K26L', 'E105H', 'E653A', 'D190I', 'I568Y',
       'Q259R', 'C630A', 'G397S', 'D176C', 'V1S', 'R387Y', 'I562V',
       'E135Q', 'K543N', 'S205V', 'R362M', 'A97I', 'P555L', 'M127R',
       'P453Y', 'A523W', 'T68M', 'P66K', 'R344W', 'L597I', 'D103S',
       'A588C', 'L407W', 'G397Q', 'D190Y', 'D209N', 'A367G', 'R518V',
       'L155M', 'H142M', 'L155I', 'T106W', 'P66Y', 'L438W', 'D281W',
       'S271W', 'L44T', 'P453S', 'D622C', 'S20D', 'D281P', 'F59G', 'F64G',
       'E184A', 'Y406N', 'K410Y', 'L471K', 'I137V', 'S426R', 'P38C',
       'V618H', 'F434P', 'I608A', 'P34M', 'H353C', 'A461W', 'R43T',
       'S218V', 'G260I', 'R324K', 'L264R', 'V361Y', 'L210Q', 'L134K',
       'R534A'], dtype=object)

In [19]:
with open('query.txt', 'w') as f:
  for mutant in querys:
    f.write(mutant+'\n')

In [50]:
#beginning amino acid embedding model

import pandas as pd

train = pd.read_csv("Data/train.csv")
test = pd.read_csv("Data/test.csv")

print(train.head())
print(len(train), len(test))

  mutant  DMS_score
0    M0Y     0.2730
1    M0W     0.2857
2    M0V     0.2153
3    M0T     0.3122
4    M0S     0.2180
1140 11324


In [51]:
def test_partition(mut):
    wt = mut[0]
    pos = int(mut[1:-1])
    mut_aa = mut[-1]
    return wt, pos, mut_aa

In [52]:
#test mutation parsing
test_partition("M0Y")

('M', 0, 'Y')

In [53]:
import numpy as np

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {aa:i for i,aa in enumerate(AMINO_ACIDS)}

In [54]:
aa_to_idx["M"]

10

In [55]:
def encode_mutation(mut):
    
    wt, pos, mut_aa = test_partition(mut)
    
    wt_idx = aa_to_idx[wt]
    mut_idx = aa_to_idx[mut_aa]
    
    return np.array([wt_idx, mut_idx, pos])

In [56]:
X = np.array([encode_mutation(m) for m in train["mutant"]])
y = train["DMS_score"].values

In [57]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=200)
model.fit(X, y)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [58]:
model.fit(X, y)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [59]:
X_test = np.array([encode_mutation(m) for m in test["mutant"]])

In [60]:
print(X_test.shape)

(11324, 3)


In [61]:
preds = model.predict(X_test)

In [62]:
print(preds[:5])

[0.622957  0.5452845 0.4561905 0.4509325 0.4847145]


In [63]:
out = pd.DataFrame({
    "mutant": test["mutant"],
    "DMS_score_predicted": preds
})

out.to_csv("predictions.csv", index=False)

In [64]:
out.head()

,mutant,DMS_score_predicted
0,V1D,0.622957
1,V1Y,0.545285
2,V1C,0.456191
3,V1A,0.450932
4,V1E,0.484714


In [43]:
#Better amino acid embedding model with normalized position:
import numpy as np

#[hydrophobicity, volume, polarity]
#https://academic.oup.com/nar/article/28/1/374/2384334


aa_embeddings = {
'A': [1.8,  88.6,  8.1],
'R': [-4.5, 173.4, 10.5],
'N': [-3.5, 114.1, 11.6],
'D': [-3.5, 111.1, 13.0],
'C': [2.5,  108.5,  5.5],
'Q': [-3.5, 143.8, 10.5],
'E': [-3.5, 138.4, 12.3],
'G': [-0.4, 60.1,  9.0],
'H': [-3.2, 153.2, 10.4],
'I': [4.5,  166.7,  5.2],
'L': [3.8,  166.7,  4.9],
'K': [-3.9, 168.6, 11.3],
'M': [1.9,  162.9,  5.7],
'F': [2.8,  189.9,  5.2],
'P': [-1.6, 112.7,  8.0],
'S': [-0.8, 89.0,  9.2],
'T': [-0.7, 116.1,  8.6],
'W': [-0.9, 227.8,  5.4],
'Y': [-1.3, 193.6,  6.2],
'V': [4.2,  140.0,  5.9],
}

In [45]:
protein_length = 182
import re

def encode_mutation(mut):
    wt = mut[0]
    mut_aa = mut[-1]
    pos = int(mut[1:-1])

    wt_emb = aa_embeddings[wt]

    mut_emb = aa_embeddings[mut_aa]

    pos_norm = pos / protein_length

    return np.array(wt_emb + mut_emb + [pos_norm])

In [46]:
X = np.array([encode_mutation(m) for m in train["mutant"]])
y = train["DMS_score"].values

X_test = np.array([encode_mutation(m) for m in test["mutant"]])

In [47]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)

preds = model.predict(X_test)

In [49]:
out = pd.DataFrame({
    "mutant": test["mutant"],
    "DMS_score_predicted": preds
})

out.to_csv("aa_predictions.csv", index=False)